# RAG Powered Q&A System

This notebook implements a Retrieval-Augmented Generation (RAG) that answers domain-specific
questions using retrieved contents from local documents


Architecture:
1. Document Loading
2. Text Splitting
3. Embedding + Vector
4. Retrieval
5. LLM Generation
6. Evaluation

In [ ]:
import os, time
import shutil
from pathlib import Path

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_cohere import CohereEmbeddings
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_chroma import Chroma

from dotenv import load_dotenv
load_dotenv()

os.environ['COHERE_API_KEY'] = os.getenv('COHERE_API_KEY')
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

## 1. Document Loading
Load specific domain text-document

In [ ]:
loader = DirectoryLoader(
    "langchain_samples",
    glob="**/*.txt",
    loader_cls = TextLoader
)
document = loader.load()
len(document)

## 2. Text Splitting
Split documents into overlapping chunks for embeddings

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    add_start_index=True
)

chunks = text_splitter.split_documents(document)

# Add Chunk ID
for i, chunk in enumerate(chunks, 1):
    chunk.metadata['chunk_id'] = i
len(chunks)

## 3. Embedding and Vector Store
Convert chunks into embeddings and store in chroma database

In [ ]:
db_path = './semantic_db'

try:
    embeddings = CohereEmbeddings(
        model="embed-english-v3.0",
        cohere_api_key=os.getenv('COHERE_API_KEY')
    )
    print("✅Chunks embedded successfully")
except Exception as e:
    print(f"❌Failed to create embeddings - {e}")
    exit(1)

if Path(db_path).exists():
    print(f"Loading existing database - {db_path}")
    vectorstore = Chroma(
        persist_directory=db_path,
        embedding_function=embeddings
    )
else:
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=db_path
    )
    print(f"Vector database created and stored {db_path}") 
    vectorstore.persist()

## Retreiever
Finds relevant information when asked

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={'k':3}
)

## Prompt + LLM

In [ ]:
rag_prompt = ChatPromptTemplate.from_template("""
You are strict document-based assistant.

Answer this questions based ONLY on the following context.

Context:
{context}

Question:
{question}

Answer: Provide a clear answer based on the following context above.
If the context doesn't contain the answer simply say
'I do not have this information in this document.'
""")

llm_groq = ChatGroq(model='llama-3.1-8b-instant', temperature=0)
parser = StrOutputParser()

## Build RAG Chain

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt | llm_groq | parser
)

## 4. Evaluation
The following queries test:
1. Retrieval Accuracy
2. Context grounding
3. Summarization ability
4. Hallucination prevention

In [ ]:
test_queries = [
    "What is the company policy on remote work?",
    "How do I set up a LangChain agent?",
    "What Python frameworks are used for data science?",
    "Summarize the vacation and remote work policies.",
    "List the recommended steps for building a RAG system using LangChain."
]

for i, query in enumerate(test_queries, 1):
    print(f"\n{i}. Query: {query}")
    print("\n🔍Searching documents....")
    time.sleep(1)
    print("\n💡 Answer:\n")
    
    response = ""
    for chunk in rag_chain.invoke(query):
        print(chunk, end="", flush=True)
        response += chunk
    print()

    if "I do not have" not in response:
        print("\n*** Retrieved Sources ***")
        retrieved_docs = retriever.invoke(query)
        for i, doc in enumerate(retrieved_docs, 1):
            print(f"{i}. └ {doc.metadata['source']}")